
# 🌍 Lab 4: การประมวลผลข้อมูลด้วย NumPy และ GeoPandas
## วิชา GE 234 Basic Programming for Geographers

### 🎯 **วัตถุประสงค์**
1. เข้าใจการใช้ **NumPy** สำหรับการประมวลผลข้อมูลทางภูมิศาสตร์ เช่น ข้อมูล Raster และพิกัด
2. ใช้ **GeoPandas** ในการจัดการและวิเคราะห์ข้อมูลเวกเตอร์ เช่น **Shapefile**
3. สามารถดำเนินการทางสถิติกับข้อมูลพิกัดและชั้นข้อมูลทางภูมิศาสตร์ได้
4. สามารถใช้ NumPy และ GeoPandas ร่วมกันเพื่อวิเคราะห์ข้อมูลได้

---

## 🔹 ตัวอย่างที่ 1: ใช้ NumPy คำนวณข้อมูล NDVI จากภาพ Raster


In [1]:

import numpy as np

# สร้างข้อมูลตัวอย่างสำหรับภาพ NDVI (Normalized Difference Vegetation Index)
nir = np.array([[0.7, 0.8, 0.6], [0.9, 0.5, 0.3], [0.4, 0.7, 0.2]])
red = np.array([[0.3, 0.4, 0.2], [0.5, 0.3, 0.1], [0.2, 0.3, 0.1]])

# คำนวณค่า NDVI
ndvi = (nir - red) / (nir + red)

print("ค่า NDVI:")
print(ndvi)


ค่า NDVI:
[[0.4        0.33333333 0.5       ]
 [0.28571429 0.25       0.5       ]
 [0.33333333 0.4        0.33333333]]



## 🔹 ตัวอย่างที่ 2: ใช้ NumPy คำนวณค่าเฉลี่ยและค่ามากสุดของ NDVI


In [2]:

print(f"ค่าเฉลี่ย NDVI: {np.mean(ndvi):.2f}")
print(f"ค่า NDVI สูงสุด: {np.max(ndvi):.2f}")
print(f"ค่า NDVI ต่ำสุด: {np.min(ndvi):.2f}")


ค่าเฉลี่ย NDVI: 0.37
ค่า NDVI สูงสุด: 0.50
ค่า NDVI ต่ำสุด: 0.25



## 🔹 ตัวอย่างที่ 3: ใช้ GeoPandas โหลดและวิเคราะห์ข้อมูล Shapefile


In [3]:

import geopandas as gpd

# โหลดข้อมูลชั้นข้อมูลจังหวัดของประเทศไทย
gdf = gpd.read_file("https://data.opendevelopmentmekong.net/en/dataset/c07d8b6b-7e39-4f62-85cb-141492f0c73a/resource/9cd1a5ff-1d17-4e71-bd15-ec5ca9616049/download/mangove_2016.zip")

# แสดงข้อมูล 5 แถวแรก
print(gdf.head())


   ogc_fid  pxlval                                           geometry
0   8592.0       1  POLYGON ((102.38289 12, 102.38444 12, 102.3844...
1   8593.0       1  POLYGON ((102.37289 12, 102.37978 12, 102.3797...
2   8595.0       1  POLYGON ((102.36889 11.99556, 102.36933 11.995...
3   8596.0       1  POLYGON ((102.39333 11.99489, 102.39356 11.994...
4   8597.0       1  POLYGON ((102.77133 11.99289, 102.77178 11.992...



## 🔹 ตัวอย่างที่ 4: คำนวณพื้นที่ของแต่ละจังหวัด


In [4]:
# ตรวจสอบค่า CRS (Coordinate Reference System)
print(gdf.crs)

# แปลง CRS ไปยังระบบพิกัดเชิงเส้น (Projected CRS) เพื่อให้การคำนวณพื้นที่ถูกต้อง
# ตัวอย่างนี้ใช้ EPSG:32647 (WGS 84 / UTM zone 47N) ซึ่งเหมาะสมกับพื้นที่ประเทศไทยบางส่วน
# หากข้อมูลครอบคลุมพื้นที่กว้าง อาจเลือก Projected CRS อื่นที่เหมาะสมกว่า เช่น World Mollweide (EPSG:54009)
gdf_projected = gdf.to_crs(epsg=32647)

# คำนวณพื้นที่ของแต่ละภูมิภาค (หน่วยเป็นตารางกิโลเมตร)
gdf_projected["area_sqkm"] = gdf_projected.geometry.area / 1e6

# แสดงภูมิภาค 5 อันดับแรกที่ใหญ่ที่สุด โดยใช้ 'ogc_fid' เป็นตัวระบุ
# เนื่องจากคอลัมน์ 'PROV_NAME' ไม่มีอยู่ในชุดข้อมูลนี้
print(gdf_projected.nlargest(5, "area_sqkm")[["ogc_fid", "area_sqkm"]])

# หากต้องการดูชื่อคอลัมน์ทั้งหมดที่อยู่ใน GeoDataFrame
# print(gdf.columns)


EPSG:4326
        ogc_fid  area_sqkm
15519  335356.0   7.985443
29263  441247.0   7.619912
37941  520369.0   6.081336
36676  512565.0   5.959017
7855   168207.0   5.037423



## 🔹 ตัวอย่างที่ 5: ใช้ GeoPandas ทำ Spatial Join


In [5]:

# โหลดข้อมูลชั้นข้อมูลอำเภอ
gdf_districts = gpd.read_file("https://data.opendevelopmentmekong.net/en/dataset/c07d8b6b-7e39-4f62-85cb-141492f0c73a/resource/9cd1a5ff-1d17-4e71-bd15-ec5ca9616049/download/mangove_2016.zip")

# ทำ Spatial Join ระหว่างอำเภอกับจังหวัด
gdf_joined = gpd.sjoin(gdf_districts, gdf, how="inner", predicate="within")

# แสดงตัวอย่างข้อมูลที่เชื่อมโยงกัน
print(gdf_joined.head())


   ogc_fid_left  pxlval_left  \
0        8592.0            1   
1        8593.0            1   
2        8595.0            1   
3        8596.0            1   
4        8597.0            1   

                                            geometry  index_right  \
0  POLYGON ((102.38289 12, 102.38444 12, 102.3844...            0   
1  POLYGON ((102.37289 12, 102.37978 12, 102.3797...            1   
2  POLYGON ((102.36889 11.99556, 102.36933 11.995...            2   
3  POLYGON ((102.39333 11.99489, 102.39356 11.994...            3   
4  POLYGON ((102.77133 11.99289, 102.77178 11.992...            4   

   ogc_fid_right  pxlval_right  
0         8592.0             1  
1         8593.0             1  
2         8595.0             1  
3         8596.0             1  
4         8597.0             1  



# 📝 **กิจกรรมในแลป**

1. **แบบฝึกหัด 1**: ใช้ NumPy คำนวณค่า **Mean, Max, Min** ของค่า NDVI ในอาร์เรย์ที่สร้างขึ้นเอง
2. **แบบฝึกหัด 2**: ใช้ GeoPandas โหลด **Shapefile** ของจังหวัด และคำนวณพื้นที่ของแต่ละจังหวัด
3. **แบบฝึกหัด 3**: ใช้ GeoPandas ทำ **Spatial Join** ระหว่างข้อมูลจังหวัดและอำเภอ
4. **แบบฝึกหัด 4**: ใช้ NumPy และ GeoPandas ร่วมกันเพื่อหาข้อมูลจังหวัดที่มี NDVI เฉลี่ยสูงสุด


### แบบฝึกหัด 1 ใช้ NumPy คำนวณค่า Mean, Max, Min ของค่า NDVI

In [6]:
import numpy as np

mean_ndvi = np.mean(ndvi)
max_ndvi = np.max(ndvi)
min_ndvi = np.min(ndvi)

print(f"ค่าเฉลี่ย NDVI: {mean_ndvi:.2f}")
print(f"ค่า NDVI สูงสุด: {max_ndvi:.2f}")
print(f"ค่า NDVI ต่ำสุด: {min_ndvi:.2f}")

ค่าเฉลี่ย NDVI: 0.37
ค่า NDVI สูงสุด: 0.50
ค่า NDVI ต่ำสุด: 0.25


### แบบฝึกหัด 2 ใช้ GeoPandas โหลด Shapefile ของจังหวัด และคำนวณพื้นที่ของแต่ละจังหวัด

In [7]:
import geopandas as gpd
from IPython.display import display

province_gdf = gpd.read_file("https://data.opendevelopmentmekong.net/th/dataset/8f3fa1b8-cb5c-48c8-9fd7-d3c213ea23db/resource/1559cee4-fedc-4330-be9c-d8cf3dd75015/download/tha_admbnda_adm1_rtsd_20190221.zip")

print("ข้อมูล Shapefile จังหวัดที่โหลดมา:")
display(province_gdf.head())
print(f"ระบบพิกัดปัจจุบัน (CRS): {province_gdf.crs}")

province_gdf_projected = province_gdf.to_crs(epsg=32647)

if 'ADM1_TH' in province_gdf_projected.columns:
    province_gdf_projected['area_sqkm'] = province_gdf_projected.geometry.area / 1e6
    print("\n10 จังหวัดที่มีพื้นที่มากที่สุด:")
    display(province_gdf_projected.nlargest(10, 'area_sqkm')[['ADM1_TH', 'area_sqkm']])
elif 'ADM1_EN' in province_gdf_projected.columns:
    province_gdf_projected['area_sqkm'] = province_gdf_projected.geometry.area / 1e6
    print("\n10 จังหวัดที่มีพื้นที่มากที่สุด:")
    display(province_gdf_projected.nlargest(10, 'area_sqkm')[['ADM1_EN', 'area_sqkm']])
elif 'name' in province_gdf_projected.columns:
    province_gdf_projected['area_sqkm'] = province_gdf_projected.geometry.area / 1e6
    print("\n10 จังหวัดที่มีพื้นที่มากที่สุด:")
    display(province_gdf_projected.nlargest(10, 'area_sqkm')[['name', 'area_sqkm']])
elif 'PROV_NAME' in province_gdf_projected.columns:
    province_gdf_projected['area_sqkm'] = province_gdf_projected.geometry.area / 1e6
    print("\n10 จังหวัดที่มีพื้นที่มากที่สุด:")
    display(province_gdf_projected.nlargest(10, 'area_sqkm')[['PROV_NAME', 'area_sqkm']])
elif 'changwat_t' in province_gdf_projected.columns:
    province_gdf_projected['area_sqkm'] = province_gdf_projected.geometry.area / 1e6
    print("\n10 จังหวัดที่มีพื้นที่มากที่สุด:")
    display(province_gdf_projected.nlargest(10, 'area_sqkm')[['changwat_t', 'area_sqkm']])
else:
    print("\nไม่พบคอลัมน์ 'ADM1_TH', 'ADM1_EN', 'name', 'PROV_NAME' หรือ 'changwat_t' ในข้อมูล กรุณาตรวจสอบชื่อคอลัมน์ที่ถูกต้องสำหรับชื่อจังหวัด")

    if 'OBJECTID' in province_gdf_projected.columns:
        province_gdf_projected['area_sqkm'] = province_gdf_projected.geometry.area / 1e6
        print("\n10 จังหวัดที่มีพื้นที่มากที่สุด (ใช้ OBJECTID):")
        display(province_gdf_projected.nlargest(10, 'area_sqkm')[['OBJECTID', 'area_sqkm']])
    else:
        province_gdf_projected['area_sqkm'] = province_gdf_projected.geometry.area / 1e6
        print("\n10 จังหวัดที่มีพื้นที่มากที่สุด (ใช้ ID อื่นๆ):")
        display(province_gdf_projected.nlargest(10, 'area_sqkm').head(10))

ข้อมูล Shapefile จังหวัดที่โหลดมา:


,Shape_Leng,Shape_Area,ADM1_EN,ADM1_TH,ADM1_PCODE,ADM1_REF,ADM1ALT1EN,ADM1ALT2EN,ADM1ALT1TH,ADM1ALT2TH,ADM0_EN,ADM0_TH,ADM0_PCODE,date,validOn,validTo,geometry
0,3.927244,0.275313,Amnat Charoen,อำนาจเจริญ,TH37,None,None,None,None,None,Thailand,ประเทศไทย,TH,2019-02-18,2019-02-21,NaT,"POLYGON ((104.95982 16.28359, 104.95986 16.283..."
1,1.739908,0.079210,Ang Thong,อ่างทอง,TH15,None,None,None,None,None,Thailand,ประเทศไทย,TH,2019-02-18,2019-02-21,NaT,"POLYGON ((100.33319 14.79853, 100.33341 14.798..."
2,2.417227,0.131339,Bangkok,กรุงเทพมหานคร,TH10,None,None,None,None,None,Thailand,ประเทศไทย,TH,2019-02-18,2019-02-21,NaT,"POLYGON ((100.61389 13.95462, 100.61428 13.954..."
3,4.414998,0.340784,Bueng Kan,บึงกาฬ,TH38,None,None,None,None,None,Thailand,ประเทศไทย,TH,2019-02-18,2019-02-21,NaT,"POLYGON ((103.40497 18.44898, 103.40619 18.448..."
4,8.701860,0.844537,Buri Ram,บุรีรัมย์,TH31,None,None,None,None,None,Thailand,ประเทศไทย,TH,2019-02-18,2019-02-21,NaT,"POLYGON ((102.93029 15.79514, 102.93029 15.795..."


ระบบพิกัดปัจจุบัน (CRS): EPSG:4326

10 จังหวัดที่มีพื้นที่มากที่สุด:


,ADM1_TH,area_sqkm
9,เชียงใหม่,22159.519971
28,นครราชสีมา,20750.869676
15,กาญจนบุรี,19436.356316
68,ตาก,17266.546740
71,อุบลราชธานี,15636.863442
66,สุราษฎร์ธานี,13075.460617
22,แม่ฮ่องสอน,12765.285306
7,ชัยภูมิ,12634.334796
18,ลำปาง,12487.598360
41,เพชรบูรณ์,12395.386041


### แบบฝึกหัด 3 ใช้ GeoPandas ทำ Spatial Join ระหว่างข้อมูลจังหวัดและอำเภอ

In [8]:
import geopandas as gpd
from IPython.display import display

gdf = gpd.read_file("https://data.opendevelopmentmekong.net/lo/dataset/0073f53b-4852-4463-ba8d-32bdef6f5476/resource/28599013-ba3e-4dbc-9d74-18ad90ce0a2f/download/district_pov.zip")

print("ข้อมูลดิบ (gdf.head()):")
display(gdf.head())

gdf_districts = gpd.read_file("https://data.opendevelopmentmekong.net/lo/dataset/0073f53b-4852-4463-ba8d-32bdef6f5476/resource/28599013-ba3e-4dbc-9d74-18ad90ce0a2f/download/district_pov.zip")

gdf_joined = gpd.sjoin(gdf_districts, gdf, how="inner", predicate="within")

print("ข้อมูลที่ผ่าน Spatial Join (gdf_joined ทั้งหมด):")
display(gdf_joined)

ข้อมูลดิบ (gdf.head()):


,PCode,Province,DCode,Area,Density,Poverty_He,Poverty_Ga,Poverty_Se,District,Population,Village_Nu,Urban_popu,Improved_S,Improved_W,Using_Elec,Own_a_Phon,district_p,district_1,geometry
0,10,Vientiane Province,1001,585.0,106.507692,9.9,1.9,0.6,Phonhong,65181.0,59.0,47.3,98.1,97.2,98.8,98.8,ວຽງຈັນ,ມ. ໂພນໂຮງ,"POLYGON ((102.49 18.49491, 102.48829 18.49262,..."
1,10,Vientiane Province,1002,783.0,65.605364,9.5,1.8,0.5,Thoulakhom,53423.0,42.0,26.4,94.0,92.4,97.7,97.2,ວຽງຈັນ,ມ. ທຸລະຄົມ,"POLYGON ((102.71878 18.48928, 102.71904 18.488..."
2,10,Vientiane Province,1006,868.0,47.306452,21.1,4.4,1.4,Feuang,41253.0,44.0,28.0,90.6,78.4,98.4,97.4,ວຽງຈັນ,ມ. ເຟືອງ,"POLYGON ((102.04717 18.40845, 102.04716 18.408..."
3,10,Vientiane Province,1007,1888.0,21.033898,11.3,2.1,0.6,Xanakharm,40027.0,34.0,16.4,96.4,88.1,98.0,96.7,ວຽງຈັນ,ມ. ຊະນະຄາມ,"POLYGON ((101.79455 18.36374, 101.79195 18.349..."
4,10,Vientiane Province,1008,1562.0,13.329065,21.9,4.5,1.4,Mad,21102.0,33.0,16.2,86.7,90.4,97.1,94.6,ວຽງຈັນ,ມ. ແມດ,"POLYGON ((102.04759 18.78031, 102.04719 18.780..."


ข้อมูลที่ผ่าน Spatial Join (gdf_joined ทั้งหมด):


,PCode_left,Province_left,DCode_left,Area_left,Density_left,Poverty_He_left,Poverty_Ga_left,Poverty_Se_left,District_left,Population_left,...,District_right,Population_right,Village_Nu_right,Urban_popu_right,Improved_S_right,Improved_W_right,Using_Elec_right,Own_a_Phon_right,district_p_right,district_1_right
0,10,Vientiane Province,1001,585.0,106.507692,9.9,1.9,0.6,Phonhong,65181.0,...,Phonhong,65181.0,59.0,47.3,98.1,97.2,98.8,98.8,ວຽງຈັນ,ມ. ໂພນໂຮງ
1,10,Vientiane Province,1002,783.0,65.605364,9.5,1.8,0.5,Thoulakhom,53423.0,...,Thoulakhom,53423.0,42.0,26.4,94.0,92.4,97.7,97.2,ວຽງຈັນ,ມ. ທຸລະຄົມ
2,10,Vientiane Province,1006,868.0,47.306452,21.1,4.4,1.4,Feuang,41253.0,...,Feuang,41253.0,44.0,28.0,90.6,78.4,98.4,97.4,ວຽງຈັນ,ມ. ເຟືອງ
3,10,Vientiane Province,1007,1888.0,21.033898,11.3,2.1,0.6,Xanakharm,40027.0,...,Xanakharm,40027.0,34.0,16.4,96.4,88.1,98.0,96.7,ວຽງຈັນ,ມ. ຊະນະຄາມ
4,10,Vientiane Province,1008,1562.0,13.329065,21.9,4.5,1.4,Mad,21102.0,...,Mad,21102.0,33.0,16.2,86.7,90.4,97.1,94.6,ວຽງຈັນ,ມ. ແມດ
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
143,7,Houaphanh,706,2378.0,15.431455,39.5,9.2,3.1,Xamtay,36860.0,...,Xamtay,36860.0,90.0,16.0,65.1,94.4,77.5,95.2,ຫົວພັນ,ມ. ຊຳໃຕ້
144,7,Houaphanh,710,2056.0,7.662938,42.8,10.3,3.5,Sone,15932.0,...,Sone,15932.0,34.0,6.6,62.9,72.2,70.6,90.3,ຫົວພັນ,ມ. ຊ່ອນ
145,7,Houaphanh,703,1408.0,8.606534,29.3,5.9,1.8,Huim,12199.0,...,Huim,12199.0,35.0,13.5,69.3,99.7,95.9,91.4,ຫົວພັນ,ມ. ຮ້ຽມ
146,9,Xiengkhuang,903,2268.0,16.492945,41.5,13.2,5.9,Nonghed,37613.0,...,Nonghed,37613.0,106.0,6.1,55.4,90.8,74.2,91.8,ຊຽງຂວາງ,ມ. ໜອງແຣດ


### แบบฝึกหัด 4 ใช้ NumPy และ GeoPandas ร่วมกันเพื่อหาข้อมูลจังหวัดที่มี NDVI เฉลี่ยสูงสุด

In [10]:
import numpy as np
import geopandas as gpd
from IPython.display import display


province_url = "https://data.opendevelopmentmekong.net/th/dataset/8f3fa1b8-cb5c-48c8-9fd7-d3c213ea23db/resource/1559cee4-fedc-4330-be9c-d8cf3dd75015/download/tha_admbnda_adm1_rtsd_20190221.zip"

try:
    province_gdf = gpd.read_file(province_url)
    print("ข้อมูล Shapefile จังหวัดที่โหลดมาแล้ว:")
    display(province_gdf.head())
    print(f"ระบบพิกัดปัจจุบัน (CRS): {province_gdf.crs}")


    province_gdf_projected = province_gdf.to_crs(epsg=32647)
    print("\nข้อมูลจังหวัดหลังการแปลง CRS เป็น EPSG:32647 (projected):")
    display(province_gdf_projected.head())


    random_ndvi = np.random.uniform(low=-0.5, high=0.8, size=len(province_gdf_projected))


    province_gdf_projected['avg_ndvi'] = random_ndvi

    print("\nข้อมูลจังหวัดพร้อมค่า NDVI เฉลี่ย (สุ่ม) 5 แถวแรก:")
    display(province_gdf_projected.head())


    province_highest_ndvi = province_gdf_projected.nlargest(1, 'avg_ndvi')

    print("\nจังหวัดที่มีค่า NDVI เฉลี่ยสูงสุด (จากค่าสุ่ม):")

    if 'ADM1_TH' in province_gdf_projected.columns:
        display(province_highest_ndvi[['ADM1_TH', 'avg_ndvi']])
    elif 'ADM1_EN' in province_gdf_projected.columns:
        display(province_highest_ndvi[['ADM1_EN', 'avg_ndvi']])
    elif 'name' in province_gdf_projected.columns:
        display(province_highest_ndvi[['name', 'avg_ndvi']])
    elif 'PROV_NAME' in province_gdf_projected.columns:
        display(province_highest_ndvi[['PROV_NAME', 'avg_ndvi']])
    elif 'changwat_t' in province_gdf_projected.columns:
        display(province_highest_ndvi[['changwat_t', 'avg_ndvi']])
    else:

        print("ไม่พบคอลัมน์ชื่อจังหวัดที่รู้จักใน GeoDataFrame แต่แสดงผลลัพธ์โดยใช้คอลัมน์ที่มีอยู่:")
        display(province_highest_ndvi[['avg_ndvi']])

except Exception as e:
    print(f"เกิดข้อผิดพลาดในการโหลดหรือประมวลผลข้อมูล: {e}")
    print("โปรดตรวจสอบลิงก์หรือว่าเซลล์ก่อนหน้า (แบบฝึกหัดที่ 2) ได้รันและสร้าง 'province_gdf_projected' อย่างถูกต้องหรือไม่")

ข้อมูล Shapefile จังหวัดที่โหลดมาแล้ว:


,Shape_Leng,Shape_Area,ADM1_EN,ADM1_TH,ADM1_PCODE,ADM1_REF,ADM1ALT1EN,ADM1ALT2EN,ADM1ALT1TH,ADM1ALT2TH,ADM0_EN,ADM0_TH,ADM0_PCODE,date,validOn,validTo,geometry
0,3.927244,0.275313,Amnat Charoen,อำนาจเจริญ,TH37,None,None,None,None,None,Thailand,ประเทศไทย,TH,2019-02-18,2019-02-21,NaT,"POLYGON ((104.95982 16.28359, 104.95986 16.283..."
1,1.739908,0.079210,Ang Thong,อ่างทอง,TH15,None,None,None,None,None,Thailand,ประเทศไทย,TH,2019-02-18,2019-02-21,NaT,"POLYGON ((100.33319 14.79853, 100.33341 14.798..."
2,2.417227,0.131339,Bangkok,กรุงเทพมหานคร,TH10,None,None,None,None,None,Thailand,ประเทศไทย,TH,2019-02-18,2019-02-21,NaT,"POLYGON ((100.61389 13.95462, 100.61428 13.954..."
3,4.414998,0.340784,Bueng Kan,บึงกาฬ,TH38,None,None,None,None,None,Thailand,ประเทศไทย,TH,2019-02-18,2019-02-21,NaT,"POLYGON ((103.40497 18.44898, 103.40619 18.448..."
4,8.701860,0.844537,Buri Ram,บุรีรัมย์,TH31,None,None,None,None,None,Thailand,ประเทศไทย,TH,2019-02-18,2019-02-21,NaT,"POLYGON ((102.93029 15.79514, 102.93029 15.795..."


ระบบพิกัดปัจจุบัน (CRS): EPSG:4326

ข้อมูลจังหวัดหลังการแปลง CRS เป็น EPSG:32647 (projected):


,Shape_Leng,Shape_Area,ADM1_EN,ADM1_TH,ADM1_PCODE,ADM1_REF,ADM1ALT1EN,ADM1ALT2EN,ADM1ALT1TH,ADM1ALT2TH,ADM0_EN,ADM0_TH,ADM0_PCODE,date,validOn,validTo,geometry
0,3.927244,0.275313,Amnat Charoen,อำนาจเจริญ,TH37,None,None,None,None,None,Thailand,ประเทศไทย,TH,2019-02-18,2019-02-21,NaT,"POLYGON ((1137719.888 1809628.932, 1137724.275..."
1,1.739908,0.079210,Ang Thong,อ่างทอง,TH15,None,None,None,None,None,Thailand,ประเทศไทย,TH,2019-02-18,2019-02-21,NaT,"POLYGON ((643472.824 1636468.792, 643495.963 1..."
2,2.417227,0.131339,Bangkok,กรุงเทพมหานคร,TH10,None,None,None,None,None,Thailand,ประเทศไทย,TH,2019-02-18,2019-02-21,NaT,"POLYGON ((674339.84 1543299.664, 674382.336 15..."
3,4.414998,0.340784,Bueng Kan,บึงกาฬ,TH38,None,None,None,None,None,Thailand,ประเทศไทย,TH,2019-02-18,2019-02-21,NaT,"POLYGON ((965496.027 2045531.181, 965625.512 2..."
4,8.701860,0.844537,Buri Ram,บุรีรัมย์,TH31,None,None,None,None,None,Thailand,ประเทศไทย,TH,2019-02-18,2019-02-21,NaT,"POLYGON ((921217.023 1750211.599, 921217.028 1..."



ข้อมูลจังหวัดพร้อมค่า NDVI เฉลี่ย (สุ่ม) 5 แถวแรก:


,Shape_Leng,Shape_Area,ADM1_EN,ADM1_TH,ADM1_PCODE,ADM1_REF,ADM1ALT1EN,ADM1ALT2EN,ADM1ALT1TH,ADM1ALT2TH,ADM0_EN,ADM0_TH,ADM0_PCODE,date,validOn,validTo,geometry,avg_ndvi
0,3.927244,0.275313,Amnat Charoen,อำนาจเจริญ,TH37,None,None,None,None,None,Thailand,ประเทศไทย,TH,2019-02-18,2019-02-21,NaT,"POLYGON ((1137719.888 1809628.932, 1137724.275...",-0.364219
1,1.739908,0.079210,Ang Thong,อ่างทอง,TH15,None,None,None,None,None,Thailand,ประเทศไทย,TH,2019-02-18,2019-02-21,NaT,"POLYGON ((643472.824 1636468.792, 643495.963 1...",0.790050
2,2.417227,0.131339,Bangkok,กรุงเทพมหานคร,TH10,None,None,None,None,None,Thailand,ประเทศไทย,TH,2019-02-18,2019-02-21,NaT,"POLYGON ((674339.84 1543299.664, 674382.336 15...",-0.499426
3,4.414998,0.340784,Bueng Kan,บึงกาฬ,TH38,None,None,None,None,None,Thailand,ประเทศไทย,TH,2019-02-18,2019-02-21,NaT,"POLYGON ((965496.027 2045531.181, 965625.512 2...",-0.023903
4,8.701860,0.844537,Buri Ram,บุรีรัมย์,TH31,None,None,None,None,None,Thailand,ประเทศไทย,TH,2019-02-18,2019-02-21,NaT,"POLYGON ((921217.023 1750211.599, 921217.028 1...",-0.327905



จังหวัดที่มีค่า NDVI เฉลี่ยสูงสุด (จากค่าสุ่ม):


,ADM1_TH,avg_ndvi
1,อ่างทอง,0.79005
